# TD 2 · Apprendre : la backpropagation, du papier à PyTorch

**Bloc 3 : deep learning · Séance 8**

---

## Objectifs

À la fin de ce TD, vous serez capable de :

1. vérifier n'importe quel gradient sans algèbre, par finite differences, et choisir
   le pas ε ;
2. calculer les gradients de la fiche par la chain rule et lire ce qu'ils disent ;
3. implémenter et entraîner un réseau 2-4-1 en NumPy (forward pass, backward pass,
   mise à jour) ;
4. expliquer par l'expérience pourquoi l'initialisation doit être aléatoire ;
5. traduire ce réseau en PyTorch, retrouver **exactement** les gradients d'autograd,
   et mesurer l'effet de deux bugs classiques ;
6. diagnostiquer l'effet du learning rate, y compris les dead units (dying ReLU),
   et tester un remède.

## Prérequis

- TD 1 (activations, forward pass de la fiche) ;
- séance 4 (gradient descent), séance 5 (cross-entropy) ;
- cours : partie 1 du diaporama, section « backpropagation ».

## Concepts mobilisés

finite differences · chain rule · computational graph · backpropagation · batch ·
symmetry breaking · autograd · `zero_grad` · `BCEWithLogitsLoss` · learning rate ·
dead units (dying ReLU) · Leaky ReLU.

## Déroulé

| Exercice | Contenu | Durée | Type | Niveau |
|---|---|---|---|---|
| 2.1 | Vérifier un gradient sans algèbre | 12 min | compléter, expérimenter | ★★☆ |
| 2.2 | La backpropagation à la main | 15 min | calculer, compléter | ★★☆ |
| 2.3 | Le réseau NumPy, par batch | 20 min | compléter | ★★☆ |
| 2.4 | Le symmetry breaking | 8 min | expérimenter | ★☆☆ |
| 2.5 | De NumPy à PyTorch | 15 min | comparer, expérimenter | ★★☆ |
| 2.6 | Learning rate et dead units | 15 min | analyser, résoudre | ★★★ |
| | Synthèse et questions | 5 min | | |

## Résultats attendus

- Une fonction `gradient_numerique` qui retrouve un gradient connu à mieux que 10⁻⁸,
  et une erreur en forme de « U » quand ε varie de 10⁻¹ à 10⁻¹³.
- Les gradients de la fiche : ∂L/∂z₂ ≈ -0,475, une colonne entièrement nulle dans
  ∂L/∂W₁ ; après un step avec un learning rate de 0,1, la loss passe de 0,644 à
  environ 0,527.
- Un réseau NumPy qui résout XOR (accuracy 1,00, loss finale ≈ 0,000913).
- Une initialisation nulle ou constante qui bloque l'apprentissage à log 2 ≈ 0,693.
- Des gradients PyTorch identiques aux vôtres (écart 0,0) ; deux bugs silencieux
  mesurés.
- Le tableau des cinq learning rates et un remède qui fonctionne à 5,0.

## Mode d'emploi

Comme au TD 1 : complétez les cellules `# TODO`, puis exécutez la cellule de
vérification qui suit. **Ce TD construit d'abord son propre arbitre** (exercice 2.1) :
toutes les vérifications suivantes le réutilisent.

## Mise en route

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from src.data import SEED, set_seed

set_seed(SEED)
np.set_printoptions(precision=6, suppress=True)
# Quatre threads : sur des tenseurs minuscules, davantage de threads ne fait que
# ralentir (voir TD 3, où ce réglage est expliqué en détail).
torch.set_num_threads(4)

# Les quatre points de XOR, qui serviront à partir de l'exercice 2.3.
X_xor = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = np.array([0.0, 1.0, 1.0, 0.0])


def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))


# La fiche de la séance 8 : un exemple, neuf paramètres fixés à la main.
x = np.array([1.0, 0.0])
y_vrai = 1.0
PARAMS = {"W1": np.array([[0.5, -0.5], [0.5, 0.5]]), "b1": np.array([0.0, 0.1]),
          "W2": np.array([[1.0, -1.0]]), "b2": np.array([0.2])}


def perte_fiche(p):
    """Forward pass de la fiche, vu comme une fonction des paramètres."""
    a1 = np.maximum(0.0, p["W1"] @ x + p["b1"])
    a2 = sigmoide(p["W2"] @ a1 + p["b2"])
    return float(-(y_vrai * np.log(a2[0]) + (1 - y_vrai) * np.log(1 - a2[0])))


print(f"loss de la fiche au départ : {perte_fiche(PARAMS):.6f}")

---

## Exercice 2.1 · Vérifier un gradient sans algèbre

*Compléter, expérimenter · 12 min · ★★☆*

**Explication.** La dérivée est la pente de la courbe. On peut la **mesurer** en
poussant un paramètre un peu dans chaque sens :

$$ \frac{\partial L}{\partial \theta} \approx \frac{L(\theta + \varepsilon) - L(\theta - \varepsilon)}{2\varepsilon} $$

C'est lent (deux forward passes par paramètre) et imprécis, donc on n'entraîne jamais
ainsi. Mais cette mesure ne partage **aucune** algèbre avec une dérivation à la main :
c'est l'arbitre idéal.

![Finite differences centrées](../assets/diagrams/s08_networks/num_grad.png)

*Schéma de la bibliothèque du cours (provenance non établie, usage en classe).*

**Exemple.** f(w) = w³ en w = 2 : la dérivée exacte vaut 3·2² = 12.

In [ ]:
f = lambda w: w ** 3  # noqa: E731
for eps in (1e-1, 1e-3, 1e-5):
    print(f"  eps = {eps:.0e} : pente mesurée {(f(2 + eps) - f(2 - eps)) / (2 * eps):.10f}")

In [ ]:
# TODO : écrivez gradient_numerique(fonction, theta, eps) pour un tableau theta de forme
# quelconque. Pour chaque case i (utilisez np.ndindex(theta.shape)) :
#   - copiez theta deux fois, ajoutez +eps à l'une et -eps à l'autre dans la case i ;
#   - appliquez la différence centrée.
# Renvoyez un tableau de même forme que theta.


def gradient_numerique(fonction, theta, eps=1e-6):
    # TODO
    raise NotImplementedError

In [ ]:
# Vérification sur une fonction dont on connaît le gradient : somme des carrés -> 2θ.
theta_test = np.array([[1.0, -2.0], [0.5, 3.0]])
ecart = np.abs(gradient_numerique(lambda t: np.sum(t ** 2), theta_test) - 2 * theta_test).max()
print(f"  écart maximal avec le gradient exact : {ecart:.1e}   "
      f"{'OK' if ecart < 1e-8 else 'à revoir'}")

**À vous : expérimenter.** Plus ε est petit, plus la mesure est précise ? Testez.

In [ ]:
# TODO : pour f(θ) = somme de sin(θ), dont le gradient exact est cos(θ), mesurez l'erreur
# maximale de gradient_numerique pour eps = 1e-1, 1e-2, ..., 1e-13 (np.logspace), en
# theta = np.array([0.3, 1.2, -2.0]). Tracez l'erreur en fonction de eps (échelles log).

**Analyse.** L'erreur dessine un **U**. À droite (ε grand), la corde s'écarte de la
tangente : erreur de troncature, proportionnelle à ε². À gauche (ε minuscule), on
soustrait deux nombres presque égaux et les erreurs d'arrondi des flottants
(environ 10⁻¹⁶) sont divisées par 2ε : c'est l'**annulation catastrophique**. Le
meilleur compromis se situe vers ε = 10⁻⁵ ; 10⁻⁶ est un choix sûr.

**Question 2.1.** Pourquoi n'entraîne-t-on jamais un réseau avec des finite
differences, alors qu'elles sont si simples à coder ?

---

## Exercice 2.2 · La backpropagation à la main

*Calculer, compléter · 15 min · ★★☆*

**Explication.** Le forward pass est une chaîne x → z₁ → a₁ → z₂ → a₂ → L. La chain
rule dit : pour dériver L par rapport à un maillon, on multiplie les dérivées
locales le long du chemin. La **backpropagation** fait ce produit une seule fois,
en partant de la fin, et réutilise chaque résultat intermédiaire.

![Computational graph de la fiche](../assets/generated/cours_08_10_fr/f05_graphe_calcul.png)

**Le premier maillon se simplifie remarquablement.** Pour une sortie sigmoïde avec
la cross-entropy :

$$ \frac{\partial L}{\partial a_2} = \frac{a_2 - y}{a_2(1 - a_2)}, \qquad
   \frac{\partial a_2}{\partial z_2} = a_2(1 - a_2) \quad\Longrightarrow\quad
   \frac{\partial L}{\partial z_2} = a_2 - y $$

Puis, mécaniquement, chaque ligne vaut « gradient venu d'en haut × dérivée locale » :

| quantité | formule | forme |
|---|---|---|
| ∂L/∂W₂ | ∂L/∂z₂ · a₁ᵀ (produit extérieur) | (1, 2) |
| ∂L/∂b₂ | ∂L/∂z₂ | (1,) |
| ∂L/∂a₁ | W₂ᵀ · ∂L/∂z₂ | (2,) |
| ∂L/∂z₁ | ∂L/∂a₁ ⊙ 1[z₁ > 0] | (2,) |
| ∂L/∂W₁ | ∂L/∂z₁ · xᵀ (produit extérieur) | (2, 2) |
| ∂L/∂b₁ | ∂L/∂z₁ | (2,) |

**À vous.** Faites d'abord le calcul sur papier (dix minutes en binôme), puis codez
les formules.

In [ ]:
# TODO : complétez le backward pass. Le forward pass est donné.
W1, b1, W2, b2 = PARAMS["W1"], PARAMS["b1"], PARAMS["W2"], PARAMS["b2"]
z1 = W1 @ x + b1
a1 = np.maximum(0.0, z1)
z2 = W2 @ a1 + b2
a2 = sigmoide(z2)

dz2 = None   # (1,)
dW2 = None   # (1, 2)   indice : np.outer
db2 = None   # (1,)
da1 = None   # (2,)
dz1 = None   # (2,)
dW1 = None   # (2, 2)
db1 = None   # (2,)

In [ ]:
# Vérification par l'arbitre de l'exercice 2.1 : chaque gradient analytique est comparé
# à la pente mesurée en poussant le paramètre correspondant.
analytiques = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
for nom, grad in analytiques.items():
    def perte_selon(valeur, nom=nom):
        return perte_fiche({**PARAMS, nom: valeur})
    mesure = gradient_numerique(perte_selon, PARAMS[nom])
    if grad is None:
        print(f"  dL/d{nom:3s} : à compléter")
    else:
        ecart = np.abs(np.asarray(grad) - mesure).max()
        print(f"  dL/d{nom:3s} = {np.round(np.asarray(grad), 6).tolist()}   "
              f"écart {ecart:.1e}   {'OK' if ecart < 1e-7 else 'à revoir'}")

In [ ]:
# Un step de gradient descent, learning rate 0,1 : la loss doit baisser.
TAUX = 0.1
if any(g is None for g in analytiques.values()):
    print("  complétez d'abord le backward pass")
else:
    nouveaux = {nom: PARAMS[nom] - TAUX * np.asarray(analytiques[nom]) for nom in PARAMS}
    print(f"  loss avant le step : {perte_fiche(PARAMS):.6f}")
    print(f"  loss après le step : {perte_fiche(nouveaux):.6f}")

**Analyse.** Trois choses se lisent dans ces gradients.

1. **∂L/∂z₂ ≈ -0,475 est négatif** : le réseau a sous-estimé (0,525 au lieu de 1).
   Le signe indique à chaque paramètre dans quel sens bouger.
2. **La deuxième colonne de ∂L/∂W₁ est nulle** : ces poids multiplient x₂, qui vaut 0
   pour cet exemple. Un poids qui n'a pas influencé la sortie ne reçoit pas de
   gradient : **le gradient ne circule que sur les chemins qui ont porté du signal**.
3. **Les deux unités cachées reçoivent des gradients opposés**, parce que
   W₂ = [1, -1] leur renvoie l'erreur avec des signes opposés.

Après un seul step, la loss est passée de 0,644 à environ 0,527.

**Question 2.2.** La loss a baissé. Est-ce garanti pour n'importe quel learning
rate ? Que se passerait-il avec un learning rate de 100 ?

---

## Exercice 2.3 · Le réseau NumPy, par batch

*Compléter · 20 min · ★★☆*

**Explication.** Les mêmes formules, sur les quatre points de XOR à la fois. Deux
changements :

1. **Par batch.** `X` est de forme (n, 2) : chaque quantité gagne une première dimension
   « exemple », et les gradients sont **moyennés** sur les exemples.
2. **Stockage transposé.** W₁ est rangé en (entrées, unités) pour écrire
   `Z1 = X @ W1 + b1` naturellement.

| quantité | forme | quantité | forme |
|---|---|---|---|
| X | (n, 2) | W1 | (2, h) |
| Z1, A1 | (n, h) | W2 | (h, 1) |
| Z2, A2 | (n, 1) | b1, b2 | (h,), (1,) |

Le produit `A1.T @ dZ2` **somme** les produits extérieurs de chaque exemple : c'est la
version par batch du `np.outer` de l'exercice 2.2.

In [ ]:
# TODO : complétez forward, backward et step. L'initialisation, la loss et
# l'activation sont données.


class ReseauNumPy:
    def __init__(self, n_entrees=2, n_cachees=4, graine=SEED):
        rng = np.random.default_rng(graine)
        # Initialisation de He : écart-type sqrt(2 / nombre d'entrées de la couche).
        self.W1 = rng.normal(0, np.sqrt(2 / n_entrees), (n_entrees, n_cachees))
        self.b1 = np.zeros(n_cachees)
        self.W2 = rng.normal(0, np.sqrt(2 / n_cachees), (n_cachees, 1))
        self.b2 = np.zeros(1)

    def activation(self, Z):
        return np.maximum(0.0, Z)

    def d_activation(self, Z):
        return (Z > 0).astype(float)

    def forward(self, X):
        # TODO : calculez et gardez self.Z1, self.A1, self.Z2, self.A2 ; renvoyez self.A2
        raise NotImplementedError

    def perte(self, y):
        p = np.clip(self.A2[:, 0], 1e-12, 1 - 1e-12)
        return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

    def backward(self, X, y):
        # TODO : renvoyez (dW1, db1, dW2, db2), moyennés sur les n exemples
        raise NotImplementedError

    def step(self, grads, taux):
        # TODO : soustrayez taux * gradient à chaque paramètre
        raise NotImplementedError

In [ ]:
# Vérification : les gradients de backward() contre l'arbitre, sur les quatre points.
# Les biais cachés sont décalés à 0,05 pour la vérification : voir la remarque
# ci-dessous.
DECALAGE = 0.05


def reseau_de_test():
    r = ReseauNumPy()
    r.b1[:] = DECALAGE
    return r


reseau = reseau_de_test()
reseau.forward(X_xor)
grads = dict(zip(["W1", "b1", "W2", "b2"], reseau.backward(X_xor, y_xor)))
for nom in grads:
    def perte_selon(valeur, nom=nom):
        copie = reseau_de_test()
        setattr(copie, nom, valeur)
        copie.forward(X_xor)
        return copie.perte(y_xor)
    ecart = np.abs(grads[nom] - gradient_numerique(perte_selon, getattr(reseau, nom))).max()
    print(f"  dL/d{nom:3s} : écart {ecart:.1e}   {'OK' if ecart < 1e-7 else 'à revoir'}")

**Remarque : un arbitre peut se tromper.** Remplacez `DECALAGE = 0.05` par `0.0` et
relancez : la vérification de b₁ échoue alors que votre code est juste. Avec des
biais nuls, l'entrée (0, 0) place **toutes** les unités exactement sur le coude de
ReLU (z = 0), là où la dérivée n'existe pas. Les finite differences mesurent la
moyenne des pentes à gauche (0) et à droite (1), soit 0,5, tandis que la
backpropagation utilise la convention 1[z > 0] = 0. Les gradient checks se font donc
en des points **où la fonction est dérivable**.

In [ ]:
def entrainer(reseau, X, y, taux=0.5, epoques=4000):
    """Gradient descent sur le batch complet ; renvoie l'historique de la loss."""
    historique = []
    for _ in range(epoques):
        reseau.forward(X)
        historique.append(reseau.perte(y))
        reseau.step(reseau.backward(X, y), taux)
    return historique


reseau = ReseauNumPy()
historique = entrainer(reseau, X_xor, y_xor)
predictions = reseau.forward(X_xor)[:, 0]
print(f"  loss initiale  : {historique[0]:.4f}")
print(f"  loss finale    : {historique[-1]:.6f}")
print(f"  prédictions    : {predictions.round(4)}")
print(f"  accuracy       : {((predictions > 0.5) == y_xor).mean():.2f}")

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(historique)
ax.set(yscale="log", xlabel="epoch", ylabel="cross-entropy",
       title="XOR appris par un réseau NumPy à 4 unités cachées")
plt.tight_layout()
plt.show()

**Analyse.** Accuracy 1,00 là où la logistic regression plafonnait à 0,50, avec
une vingtaine de lignes et aucune bibliothèque de deep learning. Si votre
loss finale vaut 0,000913, votre implémentation fait **exactement** la même
arithmétique que celle de la séance 8.

**Question 2.3.** Pourquoi divise-t-on les gradients par n (moyenne) plutôt que de les
sommer ? Que faudrait-il changer au learning rate si l'on sommait ?

---

## Exercice 2.4 · Le symmetry breaking

*Expérimenter · 8 min · ★☆☆*

**Explication.** Si deux unités cachées démarrent avec les mêmes poids, elles
calculent la même chose, reçoivent le même gradient, et restent identiques pour
toujours : le réseau se comporte comme s'il n'avait qu'une unité. Vérifions-le.

In [ ]:
# TODO : entraînez trois réseaux ReseauNumPy() (learning rate 0,5, 4 000 epochs) :
#   - initialisation aléatoire (par défaut) ;
#   - W1 et W2 mis entièrement à zéro (reseau.W1[:] = 0 ...) ;
#   - W1 et W2 mis entièrement à 0,5.
# Pour chacun, affichez la loss finale, l'accuracy et le nombre d'unités cachées
# distinctes : np.unique(reseau.W1.round(6), axis=1).shape[1].

**Analyse.** Avec des poids nuls, la hidden layer sort zéro, aucun gradient ne
remonte, et la loss reste bloquée à **log 2 ≈ 0,6931** : le réseau prédit 0,5
partout, comme la logistic regression. Avec des poids constants, les quatre unités
restent des copies conformes : un réseau à une seule unité ne peut pas représenter
XOR. L'initialisation aléatoire n'est pas un rituel : c'est ce qui **brise la
symétrie**.

**Question 2.4.** Pourquoi les **biais**, eux, peuvent-ils démarrer à zéro sans
problème ?

---

## Exercice 2.5 · De NumPy à PyTorch

*Comparer, expérimenter · 15 min · ★★☆*

**Explication.** Chaque ligne de votre réseau a sa traduction :

| votre NumPy | PyTorch | rôle |
|---|---|---|
| `W1`, `b1` tirés à la main | `nn.Linear(2, 4)` | poids **et** biais |
| `X @ W1 + b1` | `couche(x)` | l'étape affine |
| `np.maximum(0, Z1)` | `nn.ReLU()` | l'activation |
| votre `perte()` | `nn.BCEWithLogitsLoss()` | cross-entropy, sigmoïde comprise |
| votre `backward()` | `perte.backward()` | **toute** la chain rule |
| votre `step()` | `optimiseur.step()` | la mise à jour |
| (rien) | `optimiseur.zero_grad()` | remettre les gradients à zéro |

`BCEWithLogitsLoss` prend des **logits** (les z₂, avant sigmoïde) et applique la
sigmoïde elle-même, de façon numériquement stable.

In [ ]:
# TODO : écrivez la version PyTorch.
#   - torch.manual_seed(SEED) puis modele = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1))
#   - perte_fn = nn.BCEWithLogitsLoss() ; optimiseur = torch.optim.SGD(modele.parameters(), lr=0.5)
#   - Xt (float32) et yt de forme (4, 1)
#   - 4 000 epochs : zero_grad, forward pass, loss, backward, step
# Affichez la loss finale et les probabilités prédites (torch.sigmoid des logits).
modele = None

In [ ]:
# Le moment de vérité : autograd sur la fiche, en float64 partout (paramètres ET
# cible), contre vos gradients de l'exercice 2.2.
t = {nom: torch.tensor(valeur, dtype=torch.float64, requires_grad=True)
     for nom, valeur in PARAMS.items()}
t_z2 = t["W2"] @ torch.relu(t["W1"] @ torch.tensor(x) + t["b1"]) + t["b2"]
t_perte = nn.functional.binary_cross_entropy_with_logits(
    t_z2, torch.tensor([y_vrai], dtype=torch.float64))
t_perte.backward()
print(f"  loss : à la main {perte_fiche(PARAMS):.12f}   autograd {t_perte.item():.12f}")
for nom, grad in analytiques.items():
    if grad is None:
        print(f"  dL/d{nom:3s} : à compléter (exercice 2.2)")
    else:
        print(f"  dL/d{nom:3s} : écart maximal {np.abs(np.asarray(grad) - t[nom].grad.numpy()).max():.1e}")

**Analyse.** L'écart est **0,0** : autograd fait, au bit près, l'arithmétique que vous
avez faite en 2.2, sur un graphe qu'il a enregistré pendant le forward pass.
`perte.backward()` n'est pas une boîte noire.

(Si vous oubliez `dtype=torch.float64` pour la cible, `torch.tensor([1.0])` est en
float32 et la loss ne coïncide plus qu'à 10⁻⁹ près : c'est de la précision, pas une
erreur de raisonnement.)

**À vous : expérimenter.** Deux bugs classiques, qui ne lèvent **aucune erreur**.

In [ ]:
# TODO : reprenez votre training loop PyTorch dans une fonction entrainer_torch(bug)
# et mesurez la loss finale, la loss maximale atteinte et l'accuracy dans trois cas :
#   bug=None             : la training loop correcte ;
#   bug="sans_zero_grad" : la ligne optimiseur.zero_grad() est supprimée ;
#   bug="double_sigmoide": on passe torch.sigmoid(logits) à BCEWithLogitsLoss.
# Avant d'exécuter, écrivez votre prédiction pour chaque cas.
predictions_ecrites = {"sans_zero_grad": "____", "double_sigmoide": "____"}

**Analyse.** Aucun des deux bugs ne lève d'erreur, et aucun n'est spectaculaire :
l'entraînement tourne, la loss descend un peu, puis stagne. Sans `zero_grad`, les
gradients de toutes les epochs s'additionnent et la mise à jour ne correspond plus à
la loss courante ; ici, la loss ne diverge pas mais reste bloquée loin de zéro.
Avec la double sigmoïde, la sortie est coincée dans (0,5 ; 0,73) avant la seconde
sigmoïde : la loss ne peut plus descendre. **Un bug silencieux ressemble à un modèle
médiocre** : c'est ce qui le rend coûteux.

**Question 2.5.** Pourquoi PyTorch **accumule**-t-il les gradients au lieu de les
remplacer ? Donnez un usage légitime.

---

## Exercice 2.6 · Learning rate et dead units

*Analyser, résoudre · 15 min · ★★★*

**Explication.** La mise à jour θ ← θ - η·∇L : le gradient donne la direction, le
learning rate η décide de la longueur du step. On entraîne le même réseau avec
η ∈ {0,001 ; 0,05 ; 0,5 ; 5 ; 50}.

**Prédisez avant d'exécuter** : pour chaque learning rate, la loss converge-t-elle,
rampe-t-elle, oscille-t-elle ou diverge-t-elle ?

In [ ]:
# TODO (1) : vos prédictions, par écrit, AVANT d'exécuter.
mes_predictions = {0.001: "____", 0.05: "____", 0.5: "____", 5.0: "____", 50.0: "____"}

# TODO (2) : pour chaque learning rate, entraînez ReseauNumPy() 4 000 epochs et relevez :
#   - la loss finale ;
#   - le nombre de dead units : unités dont la sortie A1 est nulle pour les 4 points
#     (après reseau.forward(X_xor), compter (reseau.A1.max(axis=0) == 0).sum()) ;
#   - un verdict : "diverge", "unités détruites", "rampe encore" ou "converge".

![Cinq learning rates](../assets/generated/cours_08_10_fr/f06_taux_apprentissage.png)

**Analyse.** À 5,0, la loss finale (environ 0,48) ressemble à un apprentissage
médiocre. Ce n'en est pas un : **trois unités sur quatre sont des dead units**. Les
premiers steps, trop grands, les ont poussées du côté négatif pour tous les exemples ; elles ne
reçoivent plus de gradient et ne reviendront jamais. Le réseau descend ensuite
proprement vers le mieux que puisse faire une seule unité. Cette panne ressemble
exactement à de l'underfitting, et appelle le remède **inverse**.

**À vous : résoudre.** Réparez l'entraînement à 5,0 de deux façons.

In [ ]:
# TODO :
#  (a) remède 1 : quel learning rate choisir ? Vérifiez.
#  (b) remède 2 : créez ReseauLeaky(ReseauNumPy) qui remplace activation et
#      d_activation par Leaky ReLU (alpha = 0,1), puis entraînez-le à 5,0 et à 50,0.
#      Comptez les unités « éteintes » : (reseau.Z1.max(axis=0) <= 0).sum().

**Analyse.** Baisser le learning rate règle le problème à la source. Leaky ReLU le
règle aussi à 5,0 : une unité éteinte reçoit encore un gradient (facteur α) et peut
revenir. Mais aucun remède d'activation ne sauve un learning rate de 50 : le step
est simplement trop grand.
**Trop petit gaspille du temps ; trop grand gaspille le modèle.**

**Question 2.6.** Face à une loss finale de 0,48 sans autre information, quelles
corrections seraient tentantes, et pourquoi seraient-elles inutiles ici ? Quelle
mesure aurait permis de trancher ?

---

## Synthèse

- Les **finite differences** sont un arbitre lent mais indépendant de toute algèbre :
  ε ≈ 10⁻⁶, jamais 10⁻¹².
- La **backpropagation** est la chain rule appliquée une fois depuis la fin,
  avec réutilisation des intermédiaires. Pour sigmoïde et cross-entropy,
  ∂L/∂z = a - y.
- Le gradient ne circule que sur les chemins qui ont porté du signal (colonne nulle).
- Une initialisation nulle ou constante **ne brise pas la symétrie** : apprentissage
  bloqué.
- `loss.backward()` reproduit **exactement** vos calculs ; `zero_grad()` et les logits
  sont les deux pièges silencieux.
- Un learning rate trop grand peut **tuer des unités** : la courbe ressemble alors
  à de l'underfitting. Compter les unités vivantes avant de conclure.

## Questions de compréhension

1. Dérivez ∂L/∂z₂ = a₂ - y pour une sortie sigmoïde et la cross-entropy.
2. Pourquoi une colonne de ∂L/∂W₁ est-elle exactement nulle sur la fiche ?
3. Pourquoi l'erreur des finite differences remonte-t-elle quand ε devient minuscule ?
4. Que fait `optimizer.zero_grad()` et que se passe-t-il si on l'oublie ?
5. Comment distinguer un réseau trop petit d'un réseau aux dead units ?

## Pour aller plus loin

- Rumelhart, D., Hinton, G. et Williams, R. (1986), *Learning representations by
  back-propagating errors*, Nature 323, 533-536.
- Goodfellow, Bengio et Courville (2016), *Deep Learning*, section 6.5
  (backpropagation). https://www.deeplearningbook.org/
- Karpathy, A. (2016), *Yes you should understand backprop*.
  https://karpathy.medium.com/yes-you-should-understand-backprop-e2f06eab496b
- Maas, A., Hannun, A. et Ng, A. (2013), *Rectifier nonlinearities improve neural
  network acoustic models*, ICML Workshop (Leaky ReLU).
- PyTorch, *A Gentle Introduction to torch.autograd*.
  https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html

**Prochain TD :** la même machinerie sur les vraies données de Barcelone.